<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/InceptionV3%2BRF(Bayesian).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 6.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer

import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

In [4]:
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_lab = np.load('/content/drive/MyDrive/Y_train_labels.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_lab = np.load('/content/drive/MyDrive/Y_test_labels.npy')

In [5]:
print("--- LABEL MAPPING KEY ---")

class_map = {
    10: ("Tree cover", "#006400"),
    20: ("Shrubland", "#ffbb22"),
    30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"),
    50: ("Built-up", "#fa0000"),
    60: ("Bare / Sparse vegetation", "#b4b4b4"),
    70: ("Snow and ice", "#f0f0f0"),
    80: ("Permanent water bodies", "#0064ff"),
    90: ("Herbaceous wetland", "#0096a0"),
}

# Create the internal mapping
unique_labels = sorted(np.unique(Y_train_lab))
label_map = {old: new for new, old in enumerate(unique_labels)}

# Sorting by the new index (0, 1, 2...) for readability
for old_id, new_id in sorted(label_map.items(), key=lambda item: item[1]):
    class_name = class_map.get(old_id, ("Unknown", ""))[0]
    print(f"New ID: {new_id}  <--  Original ID: {old_id} ({class_name})")

# Apply the mapping to create the final training/testing labels
Y_train_ready = np.array([label_map[l] for l in Y_train_lab])
Y_test_ready = np.array([label_map[l] for l in Y_test_lab])


print("\n--- DATA SHAPE VERIFICATION ---")
print(f"X_train: {X_train.shape}")
print(f"Y_train_ready: {Y_train_ready.shape}")
print(f"Unique classes in Training: {np.unique(Y_train_ready)}")

--- LABEL MAPPING KEY ---
New ID: 0  <--  Original ID: 10 (Tree cover)
New ID: 1  <--  Original ID: 20 (Shrubland)
New ID: 2  <--  Original ID: 30 (Grassland)
New ID: 3  <--  Original ID: 40 (Cropland)
New ID: 4  <--  Original ID: 50 (Built-up)
New ID: 5  <--  Original ID: 80 (Permanent water bodies)

--- DATA SHAPE VERIFICATION ---
X_train: (639, 256, 256, 7)
Y_train_ready: (639,)
Unique classes in Training: [0 1 2 3 4 5]


In [7]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
import numpy as np

# 1. Data Augmentation (Safe Flips & Brightness)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomBrightness(0.1),
])

# 2. Model Input
inputs = Input(shape=(256, 256, 7))
x = data_augmentation(inputs)

# 3. Adapter with L2 Regularization (Prevents over-reliance on specific bands)
adapter = Conv2D(3, (1, 1), padding='same', name='band_adapter')(x)

# 4. InceptionV3 Backbone
inception_base = tf.keras.applications.InceptionV3(
    include_top=False,
    weights='imagenet',
    input_shape=(256, 256, 3)
)

# 5. Connect and Pool
x = inception_base(adapter)
pooled_output = GlobalAveragePooling2D(name='feature_layer')(x)

# 6. Feature Extraction Point with Dropout (Forcing Robustness)
# We apply Dropout here so the Random Forest sees regularized features

# Head for Fine-Tuning
x_head = Dense(256, activation='relu')(pooled_output)
x_head = Dropout(0.5)(x_head)
temp_predictions = Dense(len(np.unique(Y_train_ready)), activation='softmax')(x_head)

# Model for Training (Phase 1 & 2)
trainable_model = Model(inputs=inputs, outputs=temp_predictions)

# Model for Feature Extraction (Matches the dropout path)
feature_extractor = Model(inputs=inputs, outputs=pooled_output)

print("Dynamic Hybrid Architecture defined with L2 and Dropout.")

Dynamic Hybrid Architecture defined with L2 and Dropout.


In [8]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights for class balance
weights = compute_class_weight('balanced', classes=np.unique(Y_train_ready), y=Y_train_ready)
class_weight_dict = dict(enumerate(weights))
class_weight_dict[2] = class_weight_dict[2] * 1.5

# Phase 1: Freeze Inception, Train Adapter and Head
inception_base.trainable = False
trainable_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Starting Phase 1: Warming up Adapter and Head (10 Epochs)...")
trainable_model.fit(X_train, Y_train_ready,
                    epochs=10,
                    batch_size=16,
                    validation_data=(X_test, Y_test_ready),
                    class_weight=class_weight_dict)

Starting Phase 1: Warming up Adapter and Head (10 Epochs)...
Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - accuracy: 0.1784 - loss: 2.4176 - val_accuracy: 0.0880 - val_loss: 1.9263
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 122ms/step - accuracy: 0.1659 - loss: 2.0230 - val_accuracy: 0.1840 - val_loss: 1.7977
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 113ms/step - accuracy: 0.1674 - loss: 1.9798 - val_accuracy: 0.1680 - val_loss: 1.8121
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 113ms/step - accuracy: 0.1784 - loss: 1.9624 - val_accuracy: 0.1920 - val_loss: 1.8390
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 120ms/step - accuracy: 0.1831 - loss: 1.9503 - val_accuracy: 0.2080 - val_loss: 1.8133
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 116ms/step - accuracy: 0.1862 - loss: 1.9663 - val_accuracy: 0.1360 - val_loss: 1.8137
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 117ms/step - accuracy: 0.1784 - loss: 1.9604 - val_accuracy: 0.2320 - val_loss: 1.8058
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 122m

In [9]:
# Phase 2: Ultra Fine-Tuning
inception_base.trainable = True

# Lower Learning Rate to nudge weights specifically for satellite textures
trainable_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nStarting Phase 2: Ultra Fine-tuning Inception for Satellite Data...")
trainable_model.fit(X_train, Y_train_ready,
                    epochs=10,
                    batch_size=16,
                    validation_data=(X_test, Y_test_ready),
                    class_weight=class_weight_dict)


Starting Phase 2: Ultra Fine-tuning Inception for Satellite Data...
Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - accuracy: 0.1831 - loss: 1.9503 - val_accuracy: 0.2320 - val_loss: 1.8066
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 216ms/step - accuracy: 0.1831 - loss: 1.9518 - val_accuracy: 0.2320 - val_loss: 1.8104
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 221ms/step - accuracy: 0.1831 - loss: 1.9489 - val_accuracy: 0.2320 - val_loss: 1.8066
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 226ms/step - accuracy: 0.1831 - loss: 1.9437 - val_accuracy: 0.2320 - val_loss: 1.8007
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 217ms/step - accuracy: 0.1831 - loss: 1.9469 - val_accuracy: 0.2320 - val_loss: 1.7995
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 207ms/step - accuracy: 0.1831 - loss: 1.9494 - val_accuracy: 0.2320 - val_loss: 1.7957
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 207ms/step - accuracy: 0.1831 - loss: 1.9416 - val_accuracy: 0.2320 - val_loss: 1.7928
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━

In [10]:
from sklearn.decomposition import PCA
import numpy as np

print("Step 1: Extracting Raw Spectral Means (The 'Truth')...")
# Reduces (N, 256, 256, 7) to (N, 7)
X_train_raw_means = np.mean(X_train, axis=(1, 2))
X_test_raw_means = np.mean(X_test, axis=(1, 2))

print("Step 2: Extracting Inception Features (The 'Vision')...")
X_train_inc = feature_extractor.predict(X_train, batch_size=16)
X_test_inc = feature_extractor.predict(X_test, batch_size=16)

print("Step 3: Applying PCA (Reducing 2048 -> 100 features)...")
# This prevents the Inception features from "overpowering" the spectral bands
pca = PCA(n_components=100, random_state=20)
X_train_pca = pca.fit_transform(X_train_inc)
X_test_pca = pca.transform(X_test_inc)

print("Step 4: Fusing Features...")
# Combine the 100 shape-based features with the 7 color-based features
X_train_final = np.hstack((X_train_pca, X_train_raw_means))
X_test_final = np.hstack((X_test_pca, X_test_raw_means))

print(f"\nFinal Hybrid Dataset Ready!")
print(f"Train Shape: {X_train_final.shape}") # Should be (N, 107)
print(f"Test Shape:  {X_test_final.shape}")

Step 1: Extracting Raw Spectral Means (The 'Truth')...
Step 2: Extracting Inception Features (The 'Vision')...
40/40 ━━━━━━━━━━━━━━━━━━━━ 15s 223ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 709ms/step
Step 3: Applying PCA (Reducing 2048 -> 100 features)...
Step 4: Fusing Features...

Final Hybrid Dataset Ready!
Train Shape: (639, 107)
Test Shape:  (125, 107)


In [11]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def on_step(optim_result):
    it = len(optim_result.x_iters)
    current_acc = -optim_result.func_vals[-1]
    print(f"{it:<10} | {current_acc:<10.4f} ")


In [12]:
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real
from sklearn.ensemble import RandomForestClassifier

# Updated Search Space to ensure generalization
search_space = {
    'n_estimators': Integer(200, 700),
    'max_depth': Integer(5, 15),
    'min_samples_leaf': Integer(10, 40),
    'min_samples_split': Integer(20, 60),
    'max_features': Categorical(['sqrt', 'log2']),
    'max_samples': Real(0.5, 0.8)
}

rf = RandomForestClassifier(random_state=20, class_weight='balanced', n_jobs=-1)

opt = BayesSearchCV(
    estimator=rf,
    search_spaces=search_space,
    n_iter=15,
    cv=3,
    n_jobs=-1,
    verbose=0,
    random_state=20
)

print(f"\n{'Iteration':<10} | {'Curr Accuracy':<10}")
print("-" * 40)

# CRITICAL: Passing the FUSED features (X_train_final) here
opt.fit(X_train_final, Y_train_ready, callback=on_step)

# Final Results
print(f"\nBest Parameters found: {opt.best_params_}")
print(f"Best Validation F1-Macro: {opt.best_score_:.4f}")



Iteration  | Curr F1-Macro
----------------------------------------
1          | 0.7136     
2          | 0.6301     
3          | 0.6926     
4          | 0.7322     
5          | 0.6641     
6          | 0.6601     
7          | 0.6733     
8          | 0.6504     
9          | 0.6628     
10         | 0.7378     
11         | 0.7388     
12         | 0.7338     
13         | 0.7222     
14         | 0.7175     
15         | 0.7624     
16         | 0.7125     
17         | 0.7563     
18         | 0.7515     
19         | 0.7407     
20         | 0.7668     

Best Parameters found: OrderedDict({'max_depth': 5, 'max_features': 'sqrt', 'max_samples': 0.8, 'min_samples_leaf': 10, 'min_samples_split': 20, 'n_estimators': 474})
Best Validation F1-Macro: 0.7668


In [13]:
best_rf = opt.best_estimator_
y_pred = best_rf.predict(X_test_final)

target_names = ['Tree cover', 'Shrubland', 'Grassland', 'Cropland', 'Built-up', 'Permanent water']

# Final predictions
train_preds = best_rf.predict(X_train_final)
test_preds = best_rf.predict(X_test_final)

# Calculate metrics
train_acc = accuracy_score(Y_train_ready, train_preds)
test_acc = accuracy_score(Y_test_ready, test_preds)
test_f1 = f1_score(Y_test_ready, test_preds, average='macro')

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro F1:  {test_f1:.4f}")

print(classification_report(Y_test_ready, y_pred, target_names=target_names))

Train Accuracy: 0.8701
Test Accuracy:  0.7680
Test Macro F1:  0.7179
                 precision    recall  f1-score   support

     Tree cover       0.40      1.00      0.57         6
      Shrubland       0.64      0.54      0.58        13
      Grassland       0.83      0.34      0.49        29
       Cropland       0.82      0.96      0.89        28
       Built-up       0.74      0.91      0.82        22
Permanent water       0.96      0.96      0.96        27

       accuracy                           0.77       125
      macro avg       0.73      0.79      0.72       125
   weighted avg       0.80      0.77      0.75       125

